In [ ]:
# 1) Instalacao. AutoROM baixa as ROMs apos a aceitacao explicita da licenca.
%pip install -q "gymnasium[atari,accept-rom-license]>=1.0" "ale-py>=0.10" "autorom[accept-rom-license]" "imageio[ffmpeg]" pandas matplotlib
!AutoROM --accept-license

from importlib.metadata import PackageNotFoundError, version
for pacote in ["gymnasium", "ale-py", "AutoROM", "torch", "numpy", "pandas", "matplotlib", "imageio", "imageio-ffmpeg"]:
    try:
        print(f"{pacote}: {version(pacote)}")
    except PackageNotFoundError:
        print(f"{pacote}: nao encontrado")

# DQN e Double DQN no Space Invaders

Notebook executado no Google Colab para comparar os algoritmos com 200 mil passos por modelo. Inclui pré-processamento Atari, CNN, replay, checkpoints e avaliações periódicas.


## Implementação

As observações são quatro frames 84 x 84 em tons de cinza. DQN e DDQN usam a mesma CNN e diferem somente no cálculo do alvo. O ambiente usa frame skip 4, no-op reset e sticky actions.


In [ ]:
# 2) Imports, registro dos ambientes e dispositivo.
import gc
import json
import math
import random
import shutil
import time
from pathlib import Path

import ale_py
import gymnasium as gym
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

gym.register_envs(ale_py)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## Configuração

Use `smoke` para validar a execução e `relatorio` para treinar DQN e DDQN por 200 mil passos. Resultados e checkpoints podem ser gravados no Google Drive.


In [ ]:
# 3) CONFIGURACAO QUE VOCE ALTERA
AMBIENTE = "ALE/SpaceInvaders-v5"
SEMENTES = [42]
MODO = "smoke"  # depois do teste, altere somente para "relatorio"
ALGORITMOS_RELATORIO = ["DQN", "DDQN"]  # mantenha ambos para a comparacao
STICKY_ACTION_PROB = 0.25

ORCAMENTOS = {"smoke": 10_000, "relatorio": 200_000}
ALGORITMOS = ["DQN"] if MODO == "smoke" else ALGORITMOS_RELATORIO
USAR_GOOGLE_DRIVE = MODO == "relatorio"
assert AMBIENTE == "ALE/SpaceInvaders-v5"
assert MODO in ORCAMENTOS
assert ALGORITMOS and set(ALGORITMOS) <= {"DQN", "DDQN"}

if USAR_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ_RESULTADOS = Path("/content/drive/MyDrive/atari_dqn_ddqn")
else:
    RAIZ_RESULTADOS = Path("/content/atari_dqn_ddqn")
RAIZ_RESULTADOS.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "env_id": AMBIENTE, "algorithms": ALGORITMOS, "seeds": SEMENTES, "mode": MODO,
    "total_steps": ORCAMENTOS[MODO],
    "replay_capacity": 20_000, "batch_size": 32,
    "warmup_steps": 2_000 if MODO == "smoke" else 10_000,
    "train_frequency": 4, "gamma": 0.99, "learning_rate": 1e-4,
    "target_update_steps": 10_000, "gradient_clip": 10.0,
    "epsilon_start": 1.0, "epsilon_end": 0.01,
    "epsilon_decay_steps": {"smoke": 10_000, "relatorio": 100_000}[MODO],
    "eval_interval": 5_000 if MODO == "smoke" else 50_000,
    "eval_episodes": 2 if MODO == "smoke" else 10,
    "sticky_action_probability": STICKY_ACTION_PROB, "frame_stack": 4,
    "screen_size": 84, "noop_max": 30, "reward_clip": True,
    "results_root": str(RAIZ_RESULTADOS),
}
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))
if len(ALGORITMOS) > 1:
    print("ATENCAO: DQN e DDQN serao treinados em sequencia; o tempo sera aproximadamente dobrado.")

## Ambiente e teste antes do treino

A recompensa guardada no replay e cortada para `[-1, 1]`, uma pratica comum no Atari. **O retorno de episodio registrado nos CSVs e sempre a soma da recompensa bruta**, antes do clipping. `terminated` indica terminal real e impede bootstrap; `truncated` encerra/reset o episodio, mas nao e tratado como terminal no alvo.

In [ ]:
# 4) Construcao do pipeline Atari e diagnostico.
def make_atari_env(env_id, seed, render_mode=None, sticky_action_probability=0.25):
    env = gym.make(
        env_id,
        frameskip=1,
        repeat_action_probability=sticky_action_probability,
        full_action_space=False,
        render_mode=render_mode,
    )
    env = gym.wrappers.AtariPreprocessing(
        env, noop_max=CONFIG["noop_max"], frame_skip=4, screen_size=CONFIG["screen_size"],
        terminal_on_life_loss=False, grayscale_obs=True, grayscale_newaxis=False, scale_obs=False,
    )
    if hasattr(gym.wrappers, "FrameStackObservation"):
        env = gym.wrappers.FrameStackObservation(env, stack_size=CONFIG["frame_stack"])
    else:  # Compatibilidade com Gymnasium 0.29.
        env = gym.wrappers.FrameStack(env, CONFIG["frame_stack"])
    env.action_space.seed(seed)
    return env

env_teste = make_atari_env(CONFIG["env_id"], seed=SEMENTES[0])
obs, info = env_teste.reset(seed=SEMENTES[0])
obs = np.asarray(obs)
print("Observation space:", env_teste.observation_space)
print("Shape/dtype/min/max:", obs.shape, obs.dtype, obs.min(), obs.max())
print("Numero de acoes:", env_teste.action_space.n)
assert obs.shape == (4, 84, 84), f"Shape inesperado: {obs.shape}"
assert obs.dtype == np.uint8, f"dtype inesperado: {obs.dtype}"
for _ in range(50):
    obs, reward, terminated, truncated, info = env_teste.step(env_teste.action_space.sample())
    if terminated or truncated:
        obs, info = env_teste.reset()
print("Teste de 50 passos concluido.")
env_teste.close()
del env_teste

## Rede Nature DQN e replay buffer

O replay pre-aloca `estado` e `proximo_estado` como `uint8`; com 20 mil transicoes isso usa cerca de `2 x 20.000 x 4 x 84 x 84 = 1,13 GB`, mais poucos MB para acoes/recompensas/terminais. Converter para `float32` ocorre apenas no batch de 32 na GPU. Nao aumente a capacidade sem conferir a RAM. O replay nao e duplicado entre algoritmos e sera apagado ao final de cada execucao.

In [ ]:
# 5) CNN Nature DQN e replay eficiente.
class NatureDQN(nn.Module):
    def __init__(self, observation_shape, n_actions):
        super().__init__()
        c, h, w = observation_shape
        self.features = nn.Sequential(
            nn.Conv2d(c, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
        )
        with torch.no_grad():
            feature_size = self.features(torch.zeros(1, c, h, w)).flatten(1).shape[1]
        self.head = nn.Sequential(nn.Linear(feature_size, 512), nn.ReLU(), nn.Linear(512, n_actions))

    def forward(self, x):
        x = x.float().div(255.0)
        return self.head(self.features(x).flatten(1))

class ReplayBuffer:
    def __init__(self, capacity, observation_shape, seed):
        self.capacity = capacity
        self.states = np.empty((capacity, *observation_shape), dtype=np.uint8)
        self.next_states = np.empty((capacity, *observation_shape), dtype=np.uint8)
        self.actions = np.empty(capacity, dtype=np.int64)
        self.rewards = np.empty(capacity, dtype=np.float32)
        self.terminals = np.empty(capacity, dtype=np.bool_)
        self.position = 0
        self.size = 0
        self.rng = np.random.default_rng(seed)

    def add(self, state, action, reward, next_state, terminal):
        i = self.position
        self.states[i] = np.asarray(state, dtype=np.uint8)
        self.next_states[i] = np.asarray(next_state, dtype=np.uint8)
        self.actions[i] = action
        self.rewards[i] = reward
        self.terminals[i] = terminal
        self.position = (i + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size, device):
        idx = self.rng.integers(0, self.size, size=batch_size)
        return (
            torch.as_tensor(self.states[idx], device=device),
            torch.as_tensor(self.actions[idx], device=device),
            torch.as_tensor(self.rewards[idx], device=device),
            torch.as_tensor(self.next_states[idx], device=device),
            torch.as_tensor(self.terminals[idx], dtype=torch.float32, device=device),
        )

    def __len__(self):
        return self.size

modelo_teste = NatureDQN((4, 84, 84), 6)
assert modelo_teste(torch.zeros(2, 4, 84, 84, dtype=torch.uint8)).shape == (2, 6)
del modelo_teste
print("Teste de shape da CNN concluido: batch x acoes.")

## Alvos DQN e DDQN

Para transicoes nao terminais, o DQN usa `r + gamma * max_a Q_alvo(s', a)`. O DDQN usa `a* = argmax_a Q_online(s', a)` e depois `r + gamma * Q_alvo(s', a*)`. Em ambos, `(1 - terminal)` zera o segundo termo em terminal real. A perda e Huber, e o gradiente e cortado para evitar atualizacoes explosivas.

In [ ]:
# 6) Agente com formulas DQN/DDQN explicitas.
class DQNAgent:
    def __init__(self, observation_shape, n_actions, algorithm, cfg, device):
        assert algorithm in {"DQN", "DDQN"}
        self.algorithm = algorithm
        self.n_actions = n_actions
        self.cfg = cfg
        self.device = device
        self.online = NatureDQN(observation_shape, n_actions).to(device)
        self.target = NatureDQN(observation_shape, n_actions).to(device)
        self.target.load_state_dict(self.online.state_dict())
        self.target.eval()
        self.optimizer = torch.optim.Adam(self.online.parameters(), lr=cfg["learning_rate"])

    def act(self, state, epsilon, rng):
        if rng.random() < epsilon:
            return int(rng.integers(self.n_actions))
        with torch.no_grad():
            state_t = torch.as_tensor(np.asarray(state), device=self.device).unsqueeze(0)
            return int(self.online(state_t).argmax(dim=1).item())

    def optimize(self, batch):
        states, actions, rewards, next_states, terminals = batch
        q_all = self.online(states)
        q_selected = q_all.gather(1, actions.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            if self.algorithm == "DQN":
                next_q = self.target(next_states).max(dim=1).values
            else:  # DDQN: online seleciona; target avalia.
                next_actions = self.online(next_states).argmax(dim=1, keepdim=True)
                next_q = self.target(next_states).gather(1, next_actions).squeeze(1)
            targets = rewards + self.cfg["gamma"] * (1.0 - terminals) * next_q
        loss = F.smooth_l1_loss(q_selected, targets)
        self.optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(self.online.parameters(), self.cfg["gradient_clip"])
        self.optimizer.step()
        return {"loss": float(loss.item()), "q_mean": float(q_all.mean().item()), "q_max": float(q_all.max().item())}

    def hard_update(self):
        self.target.load_state_dict(self.online.state_dict())

def epsilon_at(step, cfg):
    fraction = min(step / cfg["epsilon_decay_steps"], 1.0)
    return cfg["epsilon_start"] + fraction * (cfg["epsilon_end"] - cfg["epsilon_start"])


In [ ]:
# 7) Avaliacao sem exploracao e serializacao. O replay nao entra no checkpoint.
def evaluate_agent(agent, cfg, seed, episodes):
    env = make_atari_env(cfg["env_id"], seed=seed, sticky_action_probability=cfg["sticky_action_probability"])
    returns, lengths = [], []
    was_training = agent.online.training
    agent.online.eval()
    try:
        for episode in range(episodes):
            episode_rng = np.random.default_rng(seed + episode)
            state, _ = env.reset(seed=seed + episode)
            total_return, length = 0.0, 0
            while True:
                action = agent.act(state, epsilon=0.0, rng=episode_rng)
                state, reward, terminated, truncated, _ = env.step(action)
                total_return += float(reward)
                length += 1
                if terminated or truncated:
                    break
            returns.append(total_return)
            lengths.append(length)
    finally:
        env.close()
        agent.online.train(was_training)
    return returns, lengths

def save_checkpoint(path, agent, algorithm, seed, step, best_mean_return, cfg):
    torch.save({
        "algorithm": algorithm, "seed": seed, "step": step,
        "best_mean_return": best_mean_return, "config": cfg,
        "online_state_dict": agent.online.state_dict(),
        "target_state_dict": agent.target.state_dict(),
        "optimizer_state_dict": agent.optimizer.state_dict(),
    }, path)

def load_agent(checkpoint_path, device=DEVICE):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    cfg = checkpoint["config"]
    probe = make_atari_env(cfg["env_id"], checkpoint["seed"], sticky_action_probability=cfg["sticky_action_probability"])
    agent = DQNAgent(probe.observation_space.shape, probe.action_space.n, checkpoint["algorithm"], cfg, device)
    probe.close()
    agent.online.load_state_dict(checkpoint["online_state_dict"])
    agent.target.load_state_dict(checkpoint["target_state_dict"])
    agent.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    return agent, checkpoint

In [ ]:
# 8) Loop de treino. Cada step aqui e uma decisao apos frame_skip=4.
def train_one(algorithm, seed, base_cfg):
    cfg = dict(base_cfg)
    cfg.update({"algorithm": algorithm, "seed": seed})
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    env_slug = cfg["env_id"].split("/")[-1].replace("-", "_")
    run_dir = Path(cfg["results_root"]) / f"{env_slug}_{algorithm}_{cfg['mode']}_seed{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "configuracao.json").write_text(json.dumps(cfg, indent=2, ensure_ascii=False), encoding="utf-8")

    env = make_atari_env(cfg["env_id"], seed, sticky_action_probability=cfg["sticky_action_probability"])
    observation_shape = env.observation_space.shape
    agent = DQNAgent(observation_shape, env.action_space.n, algorithm, cfg, DEVICE)
    replay = ReplayBuffer(cfg["replay_capacity"], observation_shape, seed)
    rng = np.random.default_rng(seed)
    state, _ = env.reset(seed=seed)
    episode_return, episode_length, episode_start = 0.0, 0, time.perf_counter()
    episode_metrics, episode_rows, eval_rows, update_rows = [], [], [], []
    best_mean_return = -math.inf
    training_start = time.perf_counter()

    try:
        for step in range(1, cfg["total_steps"] + 1):
            epsilon = epsilon_at(step, cfg)
            action = agent.act(state, epsilon, rng)
            next_state, raw_reward, terminated, truncated, _ = env.step(action)
            clipped_reward = float(np.clip(raw_reward, -1.0, 1.0)) if cfg["reward_clip"] else float(raw_reward)
            replay.add(state, action, clipped_reward, next_state, terminal=terminated)
            state = next_state
            episode_return += float(raw_reward)
            episode_length += 1

            if step >= cfg["warmup_steps"] and step % cfg["train_frequency"] == 0:
                metrics = agent.optimize(replay.sample(cfg["batch_size"], DEVICE))
                metrics["step"] = step
                update_rows.append(metrics)
                episode_metrics.append(metrics)

            if step % cfg["target_update_steps"] == 0:
                agent.hard_update()

            if terminated or truncated:
                duration = time.perf_counter() - episode_start
                episode_rows.append({
                    "algorithm": algorithm, "seed": seed, "episode": len(episode_rows) + 1,
                    "step": step, "raw_return": episode_return, "episode_steps": episode_length,
                    "duration_seconds": duration, "episode_fps": episode_length / max(duration, 1e-9),
                    "global_fps": step / max(time.perf_counter() - training_start, 1e-9), "epsilon": epsilon,
                    "loss_mean": np.mean([m["loss"] for m in episode_metrics]) if episode_metrics else np.nan,
                    "q_mean": np.mean([m["q_mean"] for m in episode_metrics]) if episode_metrics else np.nan,
                    "q_max": np.max([m["q_max"] for m in episode_metrics]) if episode_metrics else np.nan,
                })
                state, _ = env.reset()
                episode_return, episode_length, episode_metrics = 0.0, 0, []
                episode_start = time.perf_counter()

            if step % cfg["eval_interval"] == 0:
                returns, lengths = evaluate_agent(agent, cfg, seed + 100_000 + step, cfg["eval_episodes"])
                mean_return = float(np.mean(returns))
                eval_rows.append({
                    "algorithm": algorithm, "seed": seed, "step": step,
                    "episodes": len(returns), "mean_raw_return": mean_return,
                    "std_raw_return": float(np.std(returns)), "min_raw_return": float(np.min(returns)),
                    "max_raw_return": float(np.max(returns)), "mean_episode_steps": float(np.mean(lengths)),
                })
                is_best = mean_return > best_mean_return
                if is_best:
                    best_mean_return = mean_return
                save_checkpoint(run_dir / f"checkpoint_step_{step}.pt", agent, algorithm, seed, step, best_mean_return, cfg)
                if is_best:
                    save_checkpoint(run_dir / "melhor_checkpoint.pt", agent, algorithm, seed, step, best_mean_return, cfg)
                pd.DataFrame(episode_rows).to_csv(run_dir / "episodios_treino.csv", index=False)
                pd.DataFrame(eval_rows).to_csv(run_dir / "avaliacoes_periodicas.csv", index=False)
                pd.DataFrame(update_rows).to_csv(run_dir / "atualizacoes.csv", index=False)
                print(f"[{algorithm} seed={seed}] step={step:,} retorno_eval={mean_return:.2f} eps={epsilon:.3f}")

        save_checkpoint(run_dir / "checkpoint_final.pt", agent, algorithm, seed, cfg["total_steps"], best_mean_return, cfg)
        pd.DataFrame(episode_rows).to_csv(run_dir / "episodios_treino.csv", index=False)
        pd.DataFrame(eval_rows).to_csv(run_dir / "avaliacoes_periodicas.csv", index=False)
        pd.DataFrame(update_rows).to_csv(run_dir / "atualizacoes.csv", index=False)
        elapsed = time.perf_counter() - training_start
        execution_summary = {
            "algorithm": algorithm, "seed": seed, "total_steps": cfg["total_steps"],
            "elapsed_seconds": elapsed, "effective_steps_per_second": cfg["total_steps"] / elapsed,
            "best_mean_raw_return": best_mean_return,
        }
        (run_dir / "resumo_execucao.json").write_text(
            json.dumps(execution_summary, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        print(f"[{algorithm} seed={seed}] concluido em {elapsed / 3600:.2f} h "
              f"({execution_summary['effective_steps_per_second']:.1f} passos/s).")
        return run_dir
    finally:
        env.close()
        del replay, agent, env
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## Executar o treino (decisao explicita)

A celula abaixo vem desabilitada. Primeiro valide o ambiente e mantenha o modo padrao seguro `smoke`. Depois defina `EXECUTAR_TREINO = True`. No modo `relatorio`, checkpoints sao salvos a cada 50 mil passos; no `smoke`, a cada 5 mil. O replay buffer **nao e salvo**: os checkpoints preservam os resultados, mas nao permitem retomar exatamente o mesmo treinamento apos uma desconexao.

In [ ]:
# 9) Altere para True somente quando estiver pronto.
EXECUTAR_TREINO = False
RUN_DIRS = []
if EXECUTAR_TREINO:
    for algoritmo in ALGORITMOS:
        for semente in SEMENTES:
            RUN_DIRS.append(train_one(algoritmo, semente, CONFIG))
    print("Treinos concluidos:", [str(p) for p in RUN_DIRS])
else:
    print("Treino NAO executado. Defina EXECUTAR_TREINO=True depois do teste smoke.")

In [ ]:
# 10) Graficos. Reconstrui RUN_DIRS automaticamente se o runtime foi reiniciado.
if not RUN_DIRS:
    env_slug = CONFIG["env_id"].split("/")[-1].replace("-", "_")
    for algoritmo in ALGORITMOS:
        for semente in SEMENTES:
            candidate = Path(CONFIG["results_root"]) / f"{env_slug}_{algoritmo}_{CONFIG['mode']}_seed{semente}"
            if (candidate / "checkpoint_final.pt").exists():
                RUN_DIRS.append(candidate)
    if RUN_DIRS:
        print("Execucoes recuperadas:", [str(p) for p in RUN_DIRS])

def plot_run(run_dir):
    run_dir = Path(run_dir)
    train = pd.read_csv(run_dir / "episodios_treino.csv")
    evaluations = pd.read_csv(run_dir / "avaliacoes_periodicas.csv")
    updates = pd.read_csv(run_dir / "atualizacoes.csv")
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    train["return_ma20"] = train["raw_return"].rolling(20, min_periods=1).mean()
    axes[0].plot(train["step"], train["raw_return"], alpha=0.25, label="episodio")
    axes[0].plot(train["step"], train["return_ma20"], label="media movel 20")
    axes[0].set(title="Treino: retorno bruto", xlabel="passos", ylabel="retorno"); axes[0].legend()
    axes[1].errorbar(evaluations["step"], evaluations["mean_raw_return"], yerr=evaluations["std_raw_return"], marker="o")
    axes[1].set(title="Avaliacao sem exploracao", xlabel="passos", ylabel="retorno bruto medio")
    axes[2].plot(updates["step"], updates["loss"].rolling(500, min_periods=1).mean(), label="loss")
    ax_q = axes[2].twinx()
    ax_q.plot(updates["step"], updates["q_mean"].rolling(500, min_periods=1).mean(), color="tab:orange", label="Q medio")
    ax_q.plot(updates["step"], updates["q_max"].rolling(500, min_periods=1).mean(), color="tab:red", alpha=0.6, label="Q maximo")
    axes[2].set(title="Loss e valores Q", xlabel="passos", ylabel="Huber loss"); ax_q.set_ylabel("Q")
    fig.suptitle(run_dir.name); fig.tight_layout()
    output = run_dir / "curvas.png"
    fig.savefig(output, dpi=160, bbox_inches="tight")
    plt.show()

if RUN_DIRS:
    for directory in RUN_DIRS:
        plot_run(directory)
    if len(RUN_DIRS) > 1:
        fig, ax = plt.subplots(figsize=(8, 5))
        for directory in RUN_DIRS:
            data = pd.read_csv(Path(directory) / "avaliacoes_periodicas.csv")
            label = f"{data['algorithm'].iloc[0]} seed={data['seed'].iloc[0]}"
            ax.plot(data["step"], data["mean_raw_return"], marker="o", label=label)
        ax.set(title="DQN versus DDQN: avaliacao", xlabel="passos", ylabel="retorno bruto medio")
        ax.legend(); ax.grid(alpha=0.25); fig.tight_layout()
        comparison_path = Path(CONFIG["results_root"]) / "comparacao_avaliacao.png"
        fig.savefig(comparison_path, dpi=160, bbox_inches="tight"); plt.show()
else:
    print("RUN_DIRS esta vazio; execute o treino ou informe manualmente os diretorios.")

In [ ]:
# 11) Avalia separadamente o melhor e o checkpoint final.
def evaluate_saved_checkpoints(run_dir):
    rows = []
    for label, filename in [("melhor", "melhor_checkpoint.pt"), ("final", "checkpoint_final.pt")]:
        path = Path(run_dir) / filename
        agent, checkpoint = load_agent(path)
        cfg = checkpoint["config"]
        returns, lengths = evaluate_agent(agent, cfg, checkpoint["seed"] + 900_000, cfg["eval_episodes"])
        for episode, (ret, length) in enumerate(zip(returns, lengths), 1):
            rows.append({"checkpoint": label, "episode": episode, "raw_return": ret, "episode_steps": length})
        print(label, "media=", np.mean(returns), "desvio=", np.std(returns))
        del agent
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    result = pd.DataFrame(rows)
    result.to_csv(Path(run_dir) / "avaliacao_melhor_vs_final.csv", index=False)
    return result

AVALIACOES_FINAIS = {}
for directory in RUN_DIRS:
    AVALIACOES_FINAIS[str(directory)] = evaluate_saved_checkpoints(directory)


In [ ]:
# 12) Video MP4 curto do melhor checkpoint, renderizado em rgb_array (sem janela humana).
def record_best_video(run_dir, max_video_steps=900):
    run_dir = Path(run_dir)
    agent, checkpoint = load_agent(run_dir / "melhor_checkpoint.pt")
    cfg, seed = checkpoint["config"], checkpoint["seed"] + 950_000
    env = make_atari_env(cfg["env_id"], seed, render_mode="rgb_array", sticky_action_probability=cfg["sticky_action_probability"])
    state, _ = env.reset(seed=seed)
    rng = np.random.default_rng(seed)
    video_path = run_dir / "melhor_checkpoint.mp4"
    writer = imageio.get_writer(video_path, fps=30, codec="libx264", quality=7)
    try:
        writer.append_data(env.render())
        for _ in range(max_video_steps):
            action = agent.act(state, epsilon=0.0, rng=rng)
            state, _, terminated, truncated, _ = env.step(action)
            writer.append_data(env.render())
            if terminated or truncated:
                break
    finally:
        writer.close()
        env.close()
    del agent
    gc.collect()
    print("Video salvo em", video_path)
    return video_path

VIDEOS = [record_best_video(directory) for directory in RUN_DIRS]

In [ ]:
# 13) Compacta cada execucao e inicia o download no Colab.
def zip_and_download(run_dir):
    run_dir = Path(run_dir)
    cfg = json.loads((run_dir / "configuracao.json").read_text(encoding="utf-8"))
    env_slug = cfg["env_id"].split("/")[-1].replace("-", "_")
    zip_base = Path("/content") / f"resultados_{env_slug}_{cfg['algorithm']}_{cfg['mode']}_seed{cfg['seed']}"
    zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=run_dir))
    print("ZIP criado:", zip_path)
    try:
        from google.colab import files
        files.download(str(zip_path))
    except ImportError:
        print("Fora do Colab: download automatico ignorado.")
    return zip_path

ZIPS = [zip_and_download(directory) for directory in RUN_DIRS]